# Community creation using MICOM and computation of interaction scores

## Setup

### Requirements

In [ ]:
import matplotlib.pyplot as plt 
from libsbml import *
import os
import re
from random import sample
from math import inf
from itertools import combinations
import pickle
from pathlib import Path
import pandas as pd
from warnings import warn
from collections import OrderedDict

from cobra.io import read_sbml_model, write_sbml_model
from cobra import Reaction, Metabolite

from reframed import Model, load_cbmodel, FBA, to_cobrapy, from_cobrapy
from reframed.core.model import AttrOrderedDict, ReactionType, Compartment, Metabolite
from reframed.core.cbmodel import CBModel, CBReaction, Gene, Protein, GPRAssociation
from reframed.cobra.medium import minimal_medium
from reframed.io.cache import ModelCache
from reframed.solvers.solution import Status
from reframed.solvers import solver_instance

from smetana.legacy import Community

#### Directories and folders

## Functions

In [ ]:
communities = None
other = None
min_growth = 0.1
max_uptake = 10

In [ ]:
def load_models_per_com(community_dict, model_cache):
    """ This functions creates all communities with their respective loaded models.
     Input: community_dict : dictionary with key = community name and value = names of assigned models
     Output: com_to_models: """
    com_to_models = {}
    
    for com_label, strains in community_dict.items():
        current_community_models = []
        
        for strain in strains:
            if strain in model_cache:
                current_community_models.append(model_cache[strain])
            else:
                print(f"Warning: {strain} not found in cache")
        
        com_to_models[com_label] = current_community_models
        
    return com_to_models

In [ ]:
# no interaction minimal medium - this is really slow -> load pickle!

def build_minimal_medium_cache(model_dict):
    cache = {}
    for name, model in model_dict.items():
        exch_reactions = []
        print(f"Processing strain: {name}")
        
        # simulate
        simu_reframed = FBA(model)
        cobra_mod = to_cobrapy(model)
        simu_cobra = cobra_mod.optimize()
        if name == "iJN746":
            exch_reactions = ex_psyr
        else:
            exch_reactions = set(model.get_exchange_reactions())
        print(exch_reactions)
        # minimal medium
        res, sol = minimal_medium(
            model, direction=-1, min_mass_weight=False, min_growth = min_growth, exchange_reactions=exch_reactions,
            max_uptake=max_uptake, max_compounds=None, n_solutions=1, 
            validate=True, abstol=1e-6, warnings=True, 
            milp=True, use_pool=False, pool_gap=None, solver=None
        )
        
        # Store all results for this strain
        cache[name] = {
            "res": res, 
            "simu_cobra": simu_cobra, 
            "simu_reframed": simu_reframed,
            "sol" : sol
        }
    return cache

In [ ]:
### set of minimal requirements per non interacting community
def mini_req_percom_nointer(models2communities_dict, minimal_requirements_nointer):
    all_communities_nointer = {}
    for com_id, models in models2communities_dict.items(): 
        reactions = set()
       # print(com_id)
        #print(models)
        for strain in models:
            #print(strain)
            if strain in minimal_requirements_nointer:
                #print(strain)
                individual_res = minimal_requirements_nointer[strain]["res"]
                formatted = {r.replace("R_EX_", "R_EX_M_") + "_pool" for r in individual_res}
                reactions.update(formatted)
            else:
                print(f"Warning: {strain} not found in minimal_requirements_nointer")
        all_communities_nointer[com_id] = {"requirements": reactions}
    return all_communities_nointer

In [ ]:
def inter_medium_per_com(model2communities_dict, minimal_requirements_nointer, models_dict, min_growth, max_uptake):
    all_communities = {}

    for com_id, models in model2communities_dict.items():
        mini_req_per_comm = set()
        community_models = []

        for strain in models:
            if strain in models_dict:
                community_models.append(models_dict[strain])
                mini_req_per_strain = minimal_requirements_nointer[strain]["res"]
                mini_req_per_comm.update(mini_req_per_strain)
            else:
                print(f"Warning: Strain {strain} not found in models_dict")

        inter_comm = Community(com_id, community_models, copy_models=False, interacting=True)
        merged_exchanges = set(inter_comm.merged.get_exchange_reactions())

        mini_req_per_comm = [
            'R_EX_M_' + reac[len('R_EX_'):].replace('_e0', '_e0_pool')
            for reac in mini_req_per_comm
        ]

        missing = set(mini_req_per_comm) - merged_exchanges
        if missing:
            print(f"[{com_id}] Missing from merged model: {len(missing)} → {missing}")
        mini_req_per_comm = [r for r in mini_req_per_comm if r in merged_exchanges]
        print(f"[{com_id}] Valid reactions passed to minimal_medium: {len(mini_req_per_comm)}")

        if not mini_req_per_comm:
            print(f"[{com_id}] Skipping — no valid exchange reactions found")
            all_communities[com_id] = {"requirements": None, "solution": None}
            continue

        result_inter, sol = minimal_medium(
            inter_comm.merged,
            exchange_reactions=list(mini_req_per_comm),
            direction=-1,
            min_mass_weight=False,
            min_growth=min_growth,
            max_uptake=max_uptake,
            max_compounds=None, n_solutions=1, validate=True, abstol=1e-6,
            warnings=True, milp=True, use_pool=False, pool_gap=None, solver=None
        )

        all_communities[com_id] = {"requirements": result_inter, "solution": sol}

    return all_communities

In [ ]:
def compute_mip_score(interacting, noninteracting):
    mip2com = {}
    for com_id in interacting:
        #noex_reac_percom = set()
        #noex_reac = all_reactions[com_id]
        if interacting[com_id]["requirements"] is not None:
            mini_req_inter = len(interacting[com_id]["requirements"])
            mini_req_nointer = len(noninteracting[com_id]["requirements"])
            mip = (mini_req_nointer - mini_req_inter) / mini_req_nointer #scale to total exchange reactions 
            mip2com[com_id] = mip
        else:
            print(f"could not compute MIP for {com_id}")
    return mip2com

In [ ]:
def create_interacting_coms(model2communities_dict, models_dict):
    interact_comms_dict = {}
    for com_id, models in model2communities_dict.items():
        #print(com_id)
        community_models = []
        for strain in models:
            if strain in models_dict:
                #print(strain)
                community_models.append(models_dict[strain])
            else:
                print(f"Warning: Strain {strain} not found in models_dict")
        inter_comm = Community(com_id, community_models, copy_models=False, interacting=True)
        #print(type(inter_comm))
        interact_comms_dict[com_id] = inter_comm

    return interact_comms_dict

In [ ]:
def mro_score(dict_of_comms, direction=-1, min_mol_weight=False, min_growth=min_growth, max_uptake=max_uptake,
              validate=False, verbose=True, use_lp=False, exclude=None):
    """
    Implements the metabolic resource overlap (MRO) score as defined in (Zelezniak et al, 2015).

    Args:
        dict_of_comms (dict): mapping of community IDs to Community objects
        direction (int): direction of uptake reactions (negative or positive, default: -1)
        min_mol_weight (bool): minimize by molecular weight of nutrients (default: False)
        min_growth (float): minimum growth rate (default: 0.1)
        max_uptake (float): maximum uptake rate per organism (default: 10)
        validate (bool): validate solution (default: False)
        verbose (bool): print warnings (default: True)
        use_lp (bool): use LP instead of MILP (default: False)
        exclude (set): metabolite IDs to exclude from scoring (default: None)

    Returns:
        tuple: (mro_score_dict, extras_dict)
            mro_score_dict maps community ID -> MRO score (float or None)
            extras_dict maps community ID -> {'community_medium': set, 'individual_media': dict}
    """
    if exclude is None:
        exclude = set()

    mro_score_dict = {}
    extras_dict = {} 
    for com_id, community in dict_of_comms.items():
        #print(community)
        scaled_max_uptake = max_uptake * len(community.organisms)

        exch_reactions = set(community.merged.get_exchange_reactions())
        if verbose:
            print(f"[{com_id}] Exchange reactions: {len(exch_reactions)}")

        medium, sol = minimal_medium(
            community.merged,
            exchange_reactions=exch_reactions,
            direction=direction,
            min_mass_weight=min_mol_weight,
            min_growth=min_growth,
            max_uptake=scaled_max_uptake,
            validate=validate,
            warnings=False,
            milp=(not use_lp)
        )

        if sol.status != Status.OPTIMAL:
            if verbose:
                warn(f'MRO: Failed to find a valid community solution for: {com_id}')
            mro_score_dict[com_id] = None
            extras_dict[com_id] = None
            continue 

        interacting_env = Environment.from_reactions(medium, max_uptake=scaled_max_uptake)
        interacting_env.apply(community.merged, inplace=True)

    
        community_medium = {x[7:-7] for x in medium} - exclude

        individual_media = {}

        solver = solver_instance(community.merged)

        for org_id in community.organisms:
            if verbose:
                print(f"  [{com_id}] Solving for organism: {org_id}")

            biomass_reaction = community.organisms_biomass_reactions[org_id]
            community.merged.biomass_reaction = biomass_reaction
            org_interacting_exch = community.organisms_exchange_reactions[org_id]

            medium_i, sol_i = minimal_medium(
                community.merged,
                exchange_reactions=org_interacting_exch,
                direction=direction,
                min_mass_weight=min_mol_weight,
                min_growth=min_growth,
                max_uptake=scaled_max_uptake,
                validate=validate,
                solver=solver,
                warnings=False,
                milp=(not use_lp)
            )

            if sol_i.status != Status.OPTIMAL:
                warn(f'MRO: Failed to find a valid solution for organism: {org_id} in community: {com_id}')
                individual_media[org_id] = set() 
                continue

            org_medium = set()
            for r in medium_i:
                met = org_interacting_exch[r].original_metabolite
                if met and len(met) > 4:
                    org_medium.add(met[2:-2])
            individual_media[org_id] = org_medium - exclude

        if len(community.organisms) >= 2:
            pairwise = {
                (o1, o2): individual_media[o1] & individual_media[o2]
                for o1, o2 in combinations(community.organisms, 2)
                if o1 in individual_media and o2 in individual_media
            }
            numerator = sum(map(len, pairwise.values())) / len(pairwise) if pairwise else 0
        else:
            pairwise = {}
            numerator = 0

        denominator = (
            sum(map(len, individual_media.values())) / len(individual_media)
            if individual_media else 0
        )

        score = numerator / denominator if denominator != 0 else None

        mro_score_dict[com_id] = score

        extras_dict[com_id] = {
            'community_medium': community_medium,
            'individual_media': individual_media,
        }

    return mro_score_dict, extras_dict